In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Update with EpiScan data

 - Contains EpiScan updating and downstream analyses.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "PepCL_Manuscript":
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find PepCL_Manuscript project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Data"
SRC_DIR = PROJECT_ROOT / "src"
MODEL_CHECKPOINTS_DIR = PROJECT_ROOT / "Model_Checkpoints"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("PROJECT_ROOT in sys.path:", str(PROJECT_ROOT) in sys.path)

PROJECT_ROOT: /home/pchati/PepCL_Manuscript
DATA_DIR: /home/pchati/PepCL_Manuscript/Data
PROJECT_ROOT in sys.path: True


In [ ]:
# imports
from Data.mhc_pseudo_seqs import mhc_seq_dict
from src.general_utils import *
from src.get_feature_data import *
from src.Model.mhcprime import *
from src.model_utils import *
from src.general_utils import *
from src.get_feature_data import *
from src.pepcl import *
from src.mhcprime_inference import *
from src.log_odds import *
from src.plotting import *

# misc
import warnings
warnings.filterwarnings("ignore")
device="cuda"

In [ ]:
# change gpu
GPU_ID = 0
# GPU_ID = 1
# GPU_ID = 2
# GPU_ID = 3

torch.cuda.set_device(GPU_ID)
print(f"Using GPU: {torch.cuda.get_device_name(GPU_ID)} (ID: {torch.cuda.current_device()})")

Using GPU: NVIDIA H100 80GB HBM3 (ID: 0)


### Get datasets

#### Format pseudosequences and get feature data

In [ ]:
# format pseudosequences
mhc_seq_dict={k.removeprefix("HLA-").replace("*","").replace(":",""):v for k,v in mhc_seq_dict.items()}

In [ ]:
# get feature table and processed features
aa_features_table_ext, descriptor_groups=load_feature_data()
processed_feature_table, corr_info = process_aa_feature_table(aa_features_table_ext)
feature_names = list(processed_feature_table.columns)  # pass to FeatureProjectionEncoder

#### MS/EpiScan train and test sets

In [ ]:
# ms train and test sets
datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"
ms_train_data=pd.read_csv(f"{datasets_dir}/ms_train_data.csv", index_col=0)
ms_test_data=pd.read_csv(f"{datasets_dir}/ms_test_data.csv", index_col=0)

In [ ]:
# episcan train and test sets
episcan_train_test_dir = DATA_DIR / "Training_Datasets" / "Processed" / "EpiScan"
episcan_r9_train_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_r9_train_set.csv", index_col=0)
episcan_r9_test_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_r9_test_set.csv", index_col=0)

episcan_rt_train_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_rt_train_set.csv", index_col=0)
episcan_rt_test_set=pd.read_csv(f"{episcan_train_test_dir}/episcan_rt_test_set.csv", index_col=0)

sd3_formatted=pd.read_csv(f"{episcan_train_test_dir}/sd3_formatted.csv", index_col=0)

In [ ]:
# merge to create r9 and r9/rt train sets

# assign domain
ms_train_data["domain_id"]=0
episcan_r9_train_set["domain_id"]=1
episcan_rt_train_set["domain_id"]=1

# for the r9 set, we will take positives from both r9 and rt and then negatives from r9
ms_r9rt_pos_r9_neg_train_set=pd.concat([ms_train_data, episcan_r9_train_set, episcan_rt_train_set.query("label == 1")])

# for the r9rt set, we will take positives from both r9 and rt and negatives from both r9 and rt
ms_r9rt_pos_r9rt_neg_train_set=pd.concat([ms_train_data, episcan_r9_train_set, episcan_rt_train_set])

# reset index
ms_r9rt_pos_r9_neg_train_set.reset_index(inplace=True, drop=True)
ms_r9rt_pos_r9rt_neg_train_set.reset_index(inplace=True, drop=True)

# print stats
print_data_stats(ms_r9rt_pos_r9_neg_train_set) # easy set
print_data_stats(ms_r9rt_pos_r9rt_neg_train_set) # hard set

Total: 39,972,816
Pos: 853,369 | 2.13%
Neg: 39,119,447 | 97.87%
N alleles: 192
Total: 40,520,899
Pos: 853,369 | 2.11%
Neg: 39,667,530 | 97.89%
N alleles: 192


In [ ]:
# match allele and length distribution between domains in train
ms_r9rt_pos_r9_neg_train_set["seq_len"]=ms_r9rt_pos_r9_neg_train_set.seq.apply(len)
ms_r9rt_pos_r9rt_neg_train_set["seq_len"]=ms_r9rt_pos_r9rt_neg_train_set.seq.apply(len)

iso_alleles=['A0201', 'A0301', 'B0801', 'B5701']

ms_r9rt_pos_r9_neg_train_set_matched_allele_length=ms_r9rt_pos_r9_neg_train_set[ms_r9rt_pos_r9_neg_train_set.allele.isin(iso_alleles)].query("seq_len == 9")

ms_r9rt_pos_r9rt_neg_train_set_matched_allele_length=ms_r9rt_pos_r9rt_neg_train_set[ms_r9rt_pos_r9rt_neg_train_set.allele.isin(iso_alleles)].query("seq_len == 9")

# reset index
ms_r9rt_pos_r9_neg_train_set_matched_allele_length.reset_index(inplace=True, drop=True)
ms_r9rt_pos_r9rt_neg_train_set_matched_allele_length.reset_index(inplace=True, drop=True)

# print stats
print_data_stats(ms_r9rt_pos_r9_neg_train_set_matched_allele_length) # easy set
print_data_stats(ms_r9rt_pos_r9rt_neg_train_set_matched_allele_length) # hard set

Total: 4,942,752
Pos: 166,505 | 3.37%
Neg: 4,776,247 | 96.63%
N alleles: 4
Total: 5,490,835
Pos: 166,505 | 3.03%
Neg: 5,324,330 | 96.97%
N alleles: 4


In [ ]:
# full processing for running
ms_r9rt_pos_r9_neg_train_set_matched_allele_length=full_process_data(ms_r9rt_pos_r9_neg_train_set_matched_allele_length, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_r9rt_pos_r9_neg_train_set_matched_allele_length=create_seq_allele_column(ms_r9rt_pos_r9_neg_train_set_matched_allele_length)

ms_r9rt_pos_r9rt_neg_train_set_matched_allele_length=full_process_data(ms_r9rt_pos_r9rt_neg_train_set_matched_allele_length, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_r9rt_pos_r9rt_neg_train_set_matched_allele_length=create_seq_allele_column(ms_r9rt_pos_r9rt_neg_train_set_matched_allele_length)

ms_test_data=full_process_data(ms_test_data, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_test_data=create_seq_allele_column(ms_test_data)

sd3_formatted=full_process_data(sd3_formatted, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
sd3_formatted=create_seq_allele_column(sd3_formatted)

episcan_r9_test_set=full_process_data(episcan_r9_test_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
episcan_r9_test_set=create_seq_allele_column(episcan_r9_test_set)

episcan_rt_test_set=full_process_data(episcan_rt_test_set, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
episcan_rt_test_set=create_seq_allele_column(episcan_rt_test_set)

#### Background datasets

In [ ]:
# get background set for scoring
chp_background_for_episcsan_processed=pd.read_csv(f"{DATA_DIR}/chp_background_for_episcsan_processed.csv", index_col=0)

In [ ]:
# human proteome
canonical_human_proteins=pd.read_csv(f"{DATA_DIR}/canonical_human_proteins.csv", index_col=0)

### Load MHCPrime

In [ ]:
# setup tokenizer
tokenizer = AminoAcidTokenizer()

In [ ]:
# set base model params
base_model_params={
        'vocab_size': tokenizer.vocab_size,
        'embed_dim': 480,
        'fc_hidden_dims': [240, 120], 
        'fc_dropout': 0.1,
        'n_heads': 16,
        'max_peptide_len': 34,
        'max_mhc_len': 34,
        "n_peptide_layers": 2,
        "n_mhc_layers": 1,
        'n_concat_layers': 3,
        'add_pos_enc': True,
        'use_segment_embedder': True,
        'dim_feedforward_multiplier': 1,
        'debugging': False,
        'fc_output_dim': 1,
        'pooling': 'cls', # cls, mean, min, max, attn, for sequence ffnn head
        "ffnn_use_layernorm": False, 
        'use_feature_table': True,
        "feature_table": processed_feature_table,
        "sequence_embedder_feature_names": feature_names,
        'sequence_embedder_feature_mode': "concat",  # replace or concat
        'sequence_embedder_tokenizer': tokenizer,
        "peptide_start_pos": 1, 
        "mhc_start_pos": 35, 
        "pos_enc_type": "sincos", 
        "seed": 42,

        # peptide conformer
        "add_peptide_conformer": True, 
        "peptide_conformer_num_layers": 2,
        "peptide_conformer_n_heads": 16,
        "peptide_conformer_conv_kernel": 5,
        "peptide_conformer_dropout": 0.0,

        # mhc conformer
        "add_mhc_conformer": False, 
        "mhc_conformer_num_layers": 2,
        "mhc_conformer_n_heads": 16,
        "mhc_conformer_conv_kernel": 5,
        "mhc_conformer_dropout": 0.0,

        # fusion conformer parameters
        "add_fusion_conformer": True, 
        "fusion_conformer_num_layers": 1,
        "fusion_conformer_n_heads": 16,
        "fusion_conformer_conv_kernel": 7,
        "fusion_conformer_dropout": 0.0,
        "fusion_conformer_mask_bos": True,
        "ff_pre_post_swiglu": False,

        # legacy init
        "legacy_init_compat": True,
}

In [ ]:
%%time
# load base model (optionally, can load model_params directly)
# run_name = "MHCPrime_Base"
# run_dir = MODEL_CHECKPOINTS_DIR / run_name
# model_dir = run_dir / "Checkpoints"

# mhcprime=init_model(base_model_params, print_check=False, print_params=True)
# mhcprime.load_state_dict(torch.load(f"{model_dir}/model_final.pt"))
# mhcprime.eval()
# print("Model loaded from checkpoint...")
# clear_all_gpu_memory()

base_model_dir = MODEL_CHECKPOINTS_DIR / "MHCPrime_Base" / "Checkpoints"
ckpt_path = base_model_dir / "model_final.pt"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

clear_all_gpu_memory()

mhcprime = init_model(
    base_model_params,
    print_check=False,
    print_params=True,
)

state_dict = torch.load(
    ckpt_path,
    map_location="cpu",
)

mhcprime.load_state_dict(state_dict, strict=True)
mhcprime.to(device)
mhcprime.eval()

print("Model loaded from checkpoint...")
print(f"Checkpoint: {ckpt_path}")
print(f"Model device: {next(mhcprime.parameters()).device}")


[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
Model loaded from checkpoint...
All GPU memory has been cleared.
Current GPU memory allocated: 0.10 GB
Current GPU memory reserved: 0.16 GB
CPU times: user 492 ms, sys: 50.8 ms, total: 543 ms
Wall time: 540 ms


### PepCL updating

#### Run PepCL

In [ ]:
pepcl_base_params=dict(
            tokenizer=tokenizer,
            device="cuda",
            steps = 200,
            output_dir = None,
            save_every_n_steps = 2000,
            seed = 42,
            num_workers = 16,
            collate_fn = collate_fn,
            num_ms_pos = 2000,
            num_nonms_pos = 2000,
            ms_neg_ratio = 1.0,
            nonms_neg_ratio = 1.0,
            auto_calibrate = True,
            calibration_steps = 1,
            ms_batch_size = 3072,
            nonms_batch_size = 3072,
            encoder_lr = 2e-4,
            decoder_lr = 2e-4,
            weight_decay = 0.01,
            warmup_ratio = 0.1,
            update_domain_embeddings = False,
            primary_task_id = 1,
            sequential_backprop = True,
            ms_loss_type = "logsmoothap",
            ms_loss_alpha = 10.0,
            ms_lambda = 1.0,
            nonms_loss_type = "logsmoothap",
            nonms_loss_alpha = 10.0,
            ft_lambda = 1.0,
            dr_lambda = 1e-5,
            dr_loss_type = "mse"
)

In [ ]:
# init models
episcan_r9_ft = copy.deepcopy(mhcprime)
episcan_r9_pepcl = copy.deepcopy(mhcprime)
episcan_r9rt_ft = copy.deepcopy(mhcprime)
episcan_r9rt_pepcl = copy.deepcopy(mhcprime)
clear_all_gpu_memory()

All GPU memory has been cleared.
Current GPU memory allocated: 0.52 GB
Current GPU memory reserved: 0.56 GB


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=episcan_r9_ft,
            train_df=ms_r9rt_pos_r9_neg_train_set_matched_allele_length,
            use_replay = False,
            **pepcl_base_params,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=4, A_eff=2.60, balance=0.649, target_pos/cycle=1536, q_pos median=335.5 [310, 555], reuse median=0.0363, reuse max=0.0471, neg_ratio=1.0
  FT: alleles=4, A_eff=2.39, balance=0.597, target_pos/cycle=1536, q_pos median=335.0 [273, 593], reuse median=0.0203, reuse max=0.0361, neg_ratio=1.0



PepCL: 100%|██████████| 200/200 [01:09<00:00,  2.89it/s, MS=0.000000, FT=0.002253, DR=0.000000]


CPU times: user 1min 14s, sys: 9.72 s, total: 1min 24s
Wall time: 1min 23s


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=episcan_r9_pepcl,
            train_df=ms_r9rt_pos_r9_neg_train_set_matched_allele_length,
            use_replay = True,
            **pepcl_base_params,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=4, A_eff=2.60, balance=0.649, target_pos/cycle=1536, q_pos median=335.5 [310, 555], reuse median=0.0363, reuse max=0.0471, neg_ratio=1.0
  FT: alleles=4, A_eff=2.39, balance=0.597, target_pos/cycle=1536, q_pos median=335.0 [273, 593], reuse median=0.0203, reuse max=0.0361, neg_ratio=1.0



PepCL: 100%|██████████| 200/200 [02:37<00:00,  1.27it/s, MS=0.006924, FT=0.005689, DR=0.003822]


CPU times: user 2min 37s, sys: 18 s, total: 2min 55s
Wall time: 2min 52s


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=episcan_r9rt_ft,
            train_df=ms_r9rt_pos_r9rt_neg_train_set_matched_allele_length,
            use_replay = False,
            **pepcl_base_params,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=4, A_eff=2.60, balance=0.649, target_pos/cycle=1536, q_pos median=335.5 [310, 555], reuse median=0.0363, reuse max=0.0471, neg_ratio=1.0
  FT: alleles=4, A_eff=2.39, balance=0.597, target_pos/cycle=1536, q_pos median=335.0 [273, 593], reuse median=0.0203, reuse max=0.0361, neg_ratio=1.0



PepCL: 100%|██████████| 200/200 [01:07<00:00,  2.96it/s, MS=0.000000, FT=0.018652, DR=0.000000]


CPU times: user 1min 13s, sys: 11.1 s, total: 1min 24s
Wall time: 1min 23s


In [ ]:
%%time
# run updated pepcl function
pepcl=PepCL(
            model=episcan_r9rt_pepcl,
            train_df=ms_r9rt_pos_r9rt_neg_train_set_matched_allele_length,
            use_replay = True,
            **pepcl_base_params,
)
training_stats=pepcl.train()

[PepCL] Auto-calibrated sampling:
  MS: alleles=4, A_eff=2.60, balance=0.649, target_pos/cycle=1536, q_pos median=335.5 [310, 555], reuse median=0.0363, reuse max=0.0471, neg_ratio=1.0
  FT: alleles=4, A_eff=2.39, balance=0.597, target_pos/cycle=1536, q_pos median=335.0 [273, 593], reuse median=0.0203, reuse max=0.0361, neg_ratio=1.0



PepCL: 100%|██████████| 200/200 [02:37<00:00,  1.27it/s, MS=0.012431, FT=0.022374, DR=0.007043]


CPU times: user 2min 38s, sys: 18.5 s, total: 2min 57s
Wall time: 2min 54s


In [ ]:
%%time
"""
Save models
"""

# save_models=True
save_models=False

models_to_save={
    "episcan_r9_ft": episcan_r9_ft,
    "episcan_r9_pepcl": episcan_r9_pepcl,
    "episcan_r9rt_ft": episcan_r9rt_ft,
    "episcan_r9rt_pepcl": episcan_r9rt_pepcl,

}

episcan_model_checkpoints_dir = MODEL_CHECKPOINTS_DIR / "EpiScan_Models"
os.makedirs(episcan_model_checkpoints_dir, exist_ok=True)

if save_models:
    for model_name, model in models_to_save.items():
        print(model_name)
        torch.save(model.state_dict(), f"{episcan_model_checkpoints_dir}/{model_name}.pt")

episcan_r9_ft
episcan_r9_pepcl
episcan_r9rt_ft
episcan_r9rt_pepcl
CPU times: user 335 ms, sys: 377 ms, total: 711 ms
Wall time: 705 ms


#### Inference

In [ ]:
# get models
base_model=copy.deepcopy(mhcprime)

base_and_ft_models={
    # base model
    # "mhcprime_base": base_model,

    # updated models
    "episcan_r9_ft": episcan_r9_ft,
    "episcan_r9_pepcl": episcan_r9_pepcl,
    "episcan_r9rt_ft": episcan_r9rt_ft,
    "episcan_r9rt_pepcl": episcan_r9rt_pepcl,
}

In [ ]:
%%time
"""
Load models (optional if running inference in a different session)
"""

load_models=False
# load_models=True

if load_models:
    base_and_ft_models={"mhcprime": base_model}
    episcan_model_checkpoints_dir = MODEL_CHECKPOINTS_DIR / "EpiScan_Models"
    for model_name in os.listdir(episcan_model_checkpoints_dir):
        print(model_name)
        if model_name.endswith(".pt"):
            model_name_cleaned=model_name.removesuffix(".pt")
            print(model_name_cleaned)
            model=init_model(base_model_params, print_check=False, print_params=True)
            model.load_state_dict(torch.load(f"{episcan_model_checkpoints_dir}/{model_name}"))
            base_and_ft_models[model_name_cleaned]=model

CPU times: user 6 µs, sys: 1e+03 ns, total: 7 µs
Wall time: 11 µs


In [ ]:
%%time
# ms_test_data
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(ms_test_data, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    ms_test_data[model_name]=output_scores.detach().cpu().tolist()

3214300it [01:15, 42339.09it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (874.3 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:09<00:00,  8.08it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:10<00:00,  8.02it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:10<00:00,  8.00it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:09<00:00,  8.06it/s]


CPU times: user 9min 51s, sys: 38.1 s, total: 10min 29s
Wall time: 10min 9s


In [ ]:
# add base model (if not already added)
base_and_ft_models["mhcprime"]=base_model

In [ ]:
%%time
# episcan_r9_test_set
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(episcan_r9_test_set, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    episcan_r9_test_set[model_name]=output_scores.detach().cpu().tolist()

40000it [00:01, 28770.38it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (10.9 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 14/14 [00:04<00:00,  3.11it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 14/14 [00:04<00:00,  3.09it/s]


...

Getting data
Loading data
Setting up model
Running inference


100%|██████████| 14/14 [00:05<00:00,  2.35it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 14/14 [00:06<00:00,  2.11it/s]


CPU times: user 9.91 s, sys: 19.2 s, total: 29.2 s
Wall time: 31.8 s


In [ ]:
%%time
# episcan_rt_test_set
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(episcan_rt_test_set, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    episcan_rt_test_set[model_name]=output_scores.detach().cpu().tolist()

40000it [00:00, 42834.87it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (10.9 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 14/14 [00:04<00:00,  3.00it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 14/14 [00:06<00:00,  2.27it/s]


...

Getting data
Loading data
Setting up model
Running inference


100%|██████████| 14/14 [00:06<00:00,  2.17it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 14/14 [00:06<00:00,  2.25it/s]


CPU times: user 9.44 s, sys: 21.5 s, total: 31 s
Wall time: 33.6 s


In [ ]:
%%time
# sd3_formatted
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(sd3_formatted, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    sd3_formatted[model_name]=output_scores.detach().cpu().tolist()

326579it [00:08, 36301.93it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (88.8 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 107/107 [00:15<00:00,  6.72it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 107/107 [00:16<00:00,  6.67it/s]


...

Getting data
Loading data
Setting up model
Running inference


100%|██████████| 107/107 [00:15<00:00,  6.73it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 107/107 [00:17<00:00,  6.29it/s]


CPU times: user 1min 15s, sys: 21 s, total: 1min 36s
Wall time: 1min 36s


In [ ]:
# delete dataset cache
data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
os.remove(data_to_run_path)

In [ ]:
%%time
"""
Save all sets back (note: they are now processed)
"""

save_dfs=False
# save_dfs=True

if save_dfs:
    # ms test set
    datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"
    ms_test_data.to_csv(f"{datasets_dir}/ms_test_data.csv")

    # episcan test sets
    episcan_train_test_dir = DATA_DIR / "Training_Datasets" / "Processed" / "EpiScan"
    episcan_r9_test_set.to_csv(f"{episcan_train_test_dir}/episcan_r9_test_set.csv")
    episcan_rt_test_set.to_csv(f"{episcan_train_test_dir}/episcan_rt_test_set.csv")
    sd3_formatted.to_csv(f"{episcan_train_test_dir}/sd3_formatted.csv")

CPU times: user 15.3 s, sys: 1.03 s, total: 16.4 s
Wall time: 16.3 s


In [ ]:
%%time
# chp_background_for_episcsan_processed
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(chp_background_for_episcsan_processed, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    chp_background_for_episcsan_processed[model_name]=output_scores.detach().cpu().tolist()

398101it [00:10, 38989.26it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (108.3 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 130/130 [00:19<00:00,  6.83it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 130/130 [00:20<00:00,  6.31it/s]


...

Getting data
Loading data
Setting up model
Running inference


100%|██████████| 130/130 [00:20<00:00,  6.24it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 130/130 [00:20<00:00,  6.39it/s]


CPU times: user 1min 30s, sys: 22.5 s, total: 1min 53s
Wall time: 1min 54s


In [ ]:
# delete dataset cache
data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
os.remove(data_to_run_path)

In [ ]:
%%time
"""
Save background set
"""
# chp_background_for_episcsan_processed.to_csv(f"{DATA_DIR}/chp_background_for_episcsan_processed.csv")

CPU times: user 1.71 s, sys: 116 ms, total: 1.83 s
Wall time: 1.83 s


### Generate log odds for all sets

In [ ]:
%%time
# generate chp background
chp_filtered_negative_peptides=generate_kmers_from_proteome(canonical_human_proteins, kmer_lengths=[8,9,10,11,12,13,14])
random_negative_peptides=generate_random_peptides(n_peptides=10, lengths=[8,9,10,11,12,13,14]) 

chp_filtered_negative_peptides_no_dup={}
for l,p in tqdm(chp_filtered_negative_peptides.items()):
    chp_filtered_negative_peptides_no_dup[l]=list(set(p))
    print("Length", l)
    print(len(chp_filtered_negative_peptides[l]))
    print(len(chp_filtered_negative_peptides_no_dup[l]))

 29%|██▊       | 2/7 [00:04<00:12,  2.47s/it]

Length 8
11250173
10300888
Length 9
11229814
10405224


 57%|█████▋    | 4/7 [00:09<00:07,  2.48s/it]

Length 10
11209455
10466109
Length 11
11189096
10509168


 71%|███████▏  | 5/7 [00:12<00:04,  2.49s/it]

Length 12
11168738
10540392


 86%|████████▌ | 6/7 [00:14<00:02,  2.50s/it]

Length 13
11148381
10563518


100%|██████████| 7/7 [00:17<00:00,  2.49s/it]

Length 14
11128026
10580237
CPU times: user 1min 1s, sys: 4.95 s, total: 1min 6s
Wall time: 1min 5s


In [ ]:
# create cleaned peptide column
ms_train_data["peptide"]=ms_train_data.seq.str.replace("$", "")
ms_r9rt_pos_r9_neg_train_set_matched_allele_length["peptide"]=ms_r9rt_pos_r9_neg_train_set_matched_allele_length.seq.str.replace("$", "")
ms_test_data["peptide"]=ms_test_data.seq.str.replace("$", "")
episcan_r9_test_set["peptide"]=episcan_r9_test_set.seq.str.replace("$", "")
episcan_rt_test_set["peptide"]=episcan_rt_test_set.seq.str.replace("$", "")

In [ ]:
%%time
# compute ms log odds

# get ref_df from pt set
ref_df=ms_train_data.copy()

# ms test data
ms_test_data=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=ms_test_data,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "ms_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)

# filter down to main sets only
ms_train_data["seq_len"]=ms_train_data.peptide.apply(len)
ref_df=ms_train_data[ms_train_data.allele.isin(["A0201", "A0301", "B0801", "B5701"])].query("seq_len == 9").copy()


# episcan_r9_test_set
episcan_r9_test_set=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=episcan_r9_test_set,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "ms_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)

# episcan_rt_test_set
episcan_rt_test_set=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=episcan_rt_test_set,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "ms_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)


# episcan background
chp_background_for_episcsan_processed=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=chp_background_for_episcsan_processed,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "ms_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)


Computing PWM log-odds (allele×len): 100%|██████████| 28/28 [00:00<00:00, 303.68it/s]


CPU times: user 2min 32s, sys: 19.8 s, total: 2min 52s
Wall time: 2min 51s


In [ ]:
%%time
# episcan r9/rt log odds only
ms_r9rt_pos_r9_neg_train_set_matched_allele_length["seq_len"]=ms_r9rt_pos_r9_neg_train_set_matched_allele_length.peptide.apply(len)
ref_df=ms_r9rt_pos_r9_neg_train_set_matched_allele_length.query("domain_id == 1 and seq_len == 9").copy()

# ms test data
ms_test_data=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=ms_test_data,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "episcan_r9rt_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)

# episcan_r9_test_set
episcan_r9_test_set=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=episcan_r9_test_set,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "episcan_r9rt_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)

# episcan_rt_test_set
episcan_rt_test_set=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=episcan_rt_test_set,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "episcan_r9rt_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)


# episcan background
chp_background_for_episcsan_processed=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=chp_background_for_episcsan_processed,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "episcan_r9rt_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)



Computing PWM log-odds (allelexlen): 100%|██████████| 28/28 [00:00<00:00, 328.73it/s]


CPU times: user 2min 12s, sys: 14.5 s, total: 2min 27s
Wall time: 2min 26s


In [ ]:
%%time
# ms and episcan r9/rt log odds
ms_r9rt_pos_r9_neg_train_set_matched_allele_length["seq_len"]=ms_r9rt_pos_r9_neg_train_set_matched_allele_length.peptide.apply(len)
ref_df=ms_r9rt_pos_r9_neg_train_set_matched_allele_length.query("seq_len == 9").copy()

# ms test data
ms_test_data=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=ms_test_data,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "ms_episcan_r9rt_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)

# episcan_r9_test_set
episcan_r9_test_set=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=episcan_r9_test_set,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "ms_episcan_r9rt_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)

# episcan_rt_test_set
episcan_rt_test_set=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=episcan_rt_test_set,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "ms_episcan_r9rt_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)

# episcan background
chp_background_for_episcsan_processed=compute_multi_allele_length_pwm_logodds_gpu(
    background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
    ref_df=ref_df.query("label == 1"), # pos ms only
    df=chp_background_for_episcsan_processed,
    allele_col = "allele",
    seq_col = "peptide",
    out_col = "ms_episcan_r9rt_log_odds",
    peptide_lengths=None,
    aa_alphabet = AA_ALPHABET_20,
    pseudocount_bg = 1.0,
    pseudocount_ref = 50.0,
    device = "cuda",
)


Computing PWM log-odds (allelexlen): 100%|██████████| 28/28 [00:00<00:00, 280.31it/s]


CPU times: user 2min 12s, sys: 15.8 s, total: 2min 28s
Wall time: 2min 27s


In [ ]:
%%time
"""
Save all sets again
"""
save_dfs=False
# save_dfs=True

if save_dfs:
    # ms test set
    datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"
    ms_test_data.to_csv(f"{datasets_dir}/ms_test_data.csv")

    # episcan test sets
    episcan_train_test_dir = DATA_DIR / "Training_Datasets" / "Processed" / "EpiScan"
    episcan_r9_test_set.to_csv(f"{episcan_train_test_dir}/episcan_r9_test_set.csv")
    episcan_rt_test_set.to_csv(f"{episcan_train_test_dir}/episcan_rt_test_set.csv")

    # episcan background
    chp_background_for_episcsan_processed.to_csv(f"{DATA_DIR}/chp_background_for_episcsan_processed.csv")

CPU times: user 20 s, sys: 1.37 s, total: 21.4 s
Wall time: 21.4 s
